# Shore Distance Filtering for Fishing Effort

This notebook demonstrates **shore distance filtering** to remove false positive fishing detections near shore and on land.

## Why Filter Shore Distance?

Fishing vessels often show "fishing-like" behavior near harbors:
- ❌ **Slow speeds** - maneuvering in harbor
- ❌ **High turning** - navigating tight spaces
- ❌ **Clustered movement** - preparing gear, loading/unloading
- ❌ **On land** - GPS errors when docked

But this is NOT actual fishing! Shore filtering removes these false positives.

## Filtering Methods:

### **Coastline Distance Filter**
- Uses high-resolution coastline shapefiles
- Regional cropping (10-100x speedup!)
- Land polygon filtering (removes points ON land)

## Key Features:
- ✅ **filter_only_fishing=True** - Only processes fishing points (2-5x faster!)
- ✅ **Regional cropping** - Crops shapefiles to region (10-100x speedup)
- ✅ **Land filtering** - Removes GPS points ON land
- ✅ **Auto-detection** - Finds column names automatically

## 1. Setup & Imports

In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import pydeck as pdk
import h3
from pathlib import Path
# Shore distance filtering
from ssfaitk.utils.shore_distance_filter import (
    add_shore_filtering,
    CoastlineDistanceFilter
)

from ssfaitk.utils.trip_file_loader import (
    load_random_trips, 
    list_available_trips,
    load_trips,
    load_trip
)
from ssfaitk.models.effort.statistical_effort import StatisticalEffortClassifier


# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

saving_output_dir = "test_outputs"
# Create output directory
output_dir = Path(saving_output_dir)
output_dir.mkdir(exist_ok=True)


print("✓ Imports successful")

✓ Imports successful


## 2. Load Fishing Predictions

Load data that already has fishing classifications (from statistical or ML classifier).

**Required columns:**
- `latitude`, `longitude`: GPS coordinates
- `is_fishing` or `activity_type`: Fishing classification
- Optional: `trip_id`, `timestamp`

In [24]:
# Load your predictions
df = load_random_trips(1)
                       
print(f"✓ Data loaded: {len(df):,} points")
print(f"\nColumns: {df.columns.tolist()}")
# Show data preview
df.head()

Found 7862 trips in /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar
Randomly selected 1 trips (seed=None)
Selected trip IDs: ['13542264']
Loading 1 trips...
Loading trip 13542264 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_13542264.parquet
Loaded 1,381 points for trip 13542264
  [1/1] Trip 13542264: 1,381 points ✓
Combining 1 trips into single DataFrame...
Combined: 1,381 total points from 1 trips


✓ Data loaded: 1,381 points

Columns: ['Time', 'Boat', 'Trip', 'Lat', 'Lng', 'Speed (M/S)', 'Range (Meters)', 'Heading', 'Boat Name', 'Community']


,Time,Boat,Trip,Lat,Lng,Speed (M/S),Range (Meters),Heading,Boat Name,Community
0,2025-05-29 02:44:22+00:00,24532.0,1.3542e+07,-4.9041,39.7397,0.00,0.0000,23.0,Kitango,"Fundo,2025-05-31 15:42:05+00,2025-05-31 15:42:..."
1,2025-05-29 02:44:23+00:00,24532.0,1.3542e+07,-4.9041,39.7398,5.54,5.5457,42.0,Kitango,"Fundo,2025-05-31 15:42:05+00,2025-05-31 15:42:..."
2,2025-05-29 02:44:26+00:00,24532.0,1.3542e+07,-4.9040,39.7398,1.10,6.4623,51.0,Kitango,"Fundo,2025-05-31 15:42:05+00,2025-05-31 15:42:..."
3,2025-05-29 02:44:31+00:00,24532.0,1.3542e+07,-4.9040,39.7398,0.66,9.4731,62.0,Kitango,"Fundo,2025-05-31 15:42:05+00,2025-05-31 15:42:..."
4,2025-05-29 02:45:34+00:00,24532.0,1.3542e+07,-4.9039,39.7401,0.66,51.0613,64.0,Kitango,"Fundo,2025-05-31 15:42:05+00,2025-05-31 15:42:..."


In [25]:
clf = StatisticalEffortClassifier()

predictions_df = clf.predict(df)

print(f"\nFishing points BEFORE filtering: {(predictions_df['is_fishing']==1).sum():,}")


Resolved columns: lat=Lat, lon=Lng, time=Time, trip=Trip
Processing 1381 points across 1 trips
Computing kinematic features...
Computing local statistics...
Computing spatial features...
Computing temporal features...
Applying statistical classification rules...
Classification complete: 27.0% classified as fishing



Fishing points BEFORE filtering: 373


## 3. Coastline Distance Filtering

Precise filtering using high-resolution coastline shapefiles.

### Features:
- **Regional cropping** - Crops shapefiles to region (10-100x faster!)
- **Land filtering** - Removes points ON land
- **filter_only_fishing=True** - Only processes fishing points (2-5x faster!)

**Note:** You need GSHHS coastline shapefiles. Download from:
https://www.ngdc.noaa.gov/mgg/shorelines/gshhs.html

In [35]:
import time

print("Applying coastline filtering...\n")
start = time.time()

# Apply coastline filtering
df_filtered = add_shore_filtering(
    predictions_df,
    region='zanzibar',  # Change to your region
    method='coastline',
    coastline_shapefile='../../data/coastline/coastline_lines_wio.shp',  # Update path
    land_shapefile='../../data/coastline/coastline_land_wio.shp',  # Update path
    min_distance_km=1,  # Filter within 0.5km of coast
    filter_land_points=True,  # Remove points ON land
    filter_only_fishing=True  # Only process fishing points (FAST!)
)

elapsed = time.time() - start

# Check results
before = (predictions_df['is_fishing'] == 1).sum()
after = (df_filtered['is_fishing_filtered'] == 1).sum()
filtered = before - after
near_shore = (df_filtered['is_near_shore'] == 1).sum()
on_land = (df_filtered['is_on_land'] == True).sum()

print(f"\n{'='*60}")
print(f"COASTLINE FILTERING RESULTS")
print(f"{'='*60}")
print(f"\nProcessing time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"\nFishing points:")
print(f"  Before:   {before:,}")
print(f"  After:    {after:,}")
print(f"  Filtered: {filtered:,} ({100*filtered/before:.1f}%)")

# Show distance statistics
print(f"\nDistance to shore (fishing points):")
fishing_distances = df_filtered[df_filtered['is_fishing']==1]['dist_to_shore_km']
print(fishing_distances.describe())

Region 'zanzibar' bbox: (-7.0, -4.5, 38.5, 40.5)
Loading: ../../data/coastline/coastline_lines_wio.shp
⚡ Cropped: 1247 → 210 features (83.2% reduction)
✓ Coastline: 210 segments
Loading land: ../../data/coastline/coastline_land_wio.shp
✓ Land: 117 polygons
Filter mode: Only processing 373 fishing points (skipping 1,008 non-fishing)
Computing distances for 373 points...
✓ On land: 0 (0.0% of processed)
✓ Skipped 1,008 non-fishing points (dist set to 999 km)
Total filtered: 55 (3.98%)


Applying coastline filtering...


COASTLINE FILTERING RESULTS

Processing time: 0.1s (0.0 min)

Fishing points:
  Before:   373
  After:    318
  Filtered: 55 (14.7%)

Distance to shore (fishing points):
count    373.0000
mean       3.1755
std        1.8256
min        0.4897
25%        1.1071
50%        4.4400
75%        4.6504
max        4.9589
Name: dist_to_shore_km, dtype: float64


## 4. Clean Data (Remove Filtered Points)

Remove near-shore and on-land points for clean visualization.

In [36]:
# Remove near-shore and on-land points
df_clean = df_filtered[df_filtered['is_near_shore'] != 1].copy()
df_clean = df_clean[~df_clean['is_on_land']].copy()

# Get fishing points only
fishing_clean = df_clean[df_clean['is_fishing'] == 1]

print(f"Clean data:")
print(f"  Total points: {len(df_clean):,}")
print(f"  Fishing points: {len(fishing_clean):,}")
print(f"  Near-shore remaining: {df_clean['is_near_shore'].sum():,}")
print(f"  On-land remaining: {df_clean['is_on_land'].sum():,}")

Clean data:
  Total points: 1,326
  Fishing points: 318
  Near-shore remaining: 0
  On-land remaining: 0


## 5. 3D Heatmap Visualization

Create interactive 3D hexagonal heatmap of fishing density using pydeck.

**Features:**
- H3 hexagonal aggregation
- 3D column bars showing density
- Color gradient: Blue (low) → Red (high)
- Dark Carto basemap
- Interactive (rotate, zoom, tilt)

In [37]:
print("Creating 3D heatmap...\n")

# Use clean fishing data
fishing = fishing_clean.copy()

print(f"Processing {len(fishing):,} fishing points...")

# Aggregate to H3 hexagons (resolution 8 = ~500m)
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

# Count per hexagon
hex_counts = fishing.groupby('h3').size().reset_index(name='count')

# Get hexagon centers
hex_counts['lat'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_counts['lon'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

print(f"Aggregated to {len(hex_counts):,} hexagons")
print(f"  Min count: {hex_counts['count'].min()}")
print(f"  Max count: {hex_counts['count'].max()}")
print(f"  Mean count: {hex_counts['count'].mean():.1f}")

# Normalize for colors (blue → cyan → yellow → red)
max_count = hex_counts['count'].max()
hex_counts['color_r'] = (hex_counts['count'] / max_count * 255).astype(int)
hex_counts['color_g'] = 255 - hex_counts['color_r']

# Create 3D column layer
layer = pdk.Layer(
    'ColumnLayer',
    data=hex_counts,
    get_position=['lon', 'lat'],
    get_elevation='count',
    elevation_scale=2,  # Adjust bar height
    radius=300,  # Hexagon size in meters
    get_fill_color='[color_r, color_g, 255, 180]',  # Blue → Red gradient
    pickable=True,
    auto_highlight=True
)

# Set view
view_state = pdk.ViewState(
    latitude=hex_counts['lat'].mean(),
    longitude=hex_counts['lon'].mean(),
    zoom=10,
    pitch=50,  # Tilt angle for 3D view
    bearing=0
)

# Create deck
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
    tooltip={'text': 'Fishing Points: {count}'}
)

# Save to HTML
output_file = saving_output_dir+'/fishing_3d_heatmap_filtered.html'
r.to_html(output_file)

print(f"\n✓ 3D heatmap saved to: {output_file}")
print(f"\nOpen the file to view interactive 3D visualization!")
print(f"Controls:")
print(f"  - Drag to pan")
print(f"  - Scroll to zoom")
print(f"  - Right-click drag to rotate")
print(f"  - Shift + drag to tilt")

Creating 3D heatmap...

Processing 318 fishing points...
Aggregated to 2 hexagons
  Min count: 88
  Max count: 230
  Mean count: 159.0

✓ 3D heatmap saved to: test_outputs/fishing_3d_heatmap_filtered.html

Open the file to view interactive 3D visualization!
Controls:
  - Drag to pan
  - Scroll to zoom
  - Right-click drag to rotate
  - Shift + drag to tilt


## 6. Save Filtered Results

In [38]:
# Save filtered data
output_path = saving_output_dir+'/predictions_shore_filtered.parquet'
df_filtered.to_parquet(output_path, index=False)

print(f"\n✓ Filtered data saved to: {output_path}")
print(f"\nColumns added:")
print(f"  - dist_to_shore_km: Distance to nearest coastline (km)")
print(f"  - is_near_shore: 1 if within threshold, 0 otherwise")
print(f"  - is_on_land: True if point is on land")
print(f"  - is_fishing_filtered: Filtered fishing classification")

# Save clean data (no near-shore/land)
clean_output = saving_output_dir+'/predictions_clean.parquet'
df_clean.to_parquet(clean_output, index=False)
print(f"\n✓ Clean data saved to: {clean_output}")


✓ Filtered data saved to: test_outputs/predictions_shore_filtered.parquet

Columns added:
  - dist_to_shore_km: Distance to nearest coastline (km)
  - is_near_shore: 1 if within threshold, 0 otherwise
  - is_on_land: True if point is on land
  - is_fishing_filtered: Filtered fishing classification

✓ Clean data saved to: test_outputs/predictions_clean.parquet
